# Kapitola 3: Přiřazování rolí (Role Prompting)

- [Lekce](#lesson)
- [Cvičení](#exercises)
- [Příkladové hřiště](#example-playground)

## Nastavení

Spusťte následující buňku pro nastavení, abyste načetli svůj API klíč a vytvořili pomocnou funkci `get_completion`.

In [ ]:
%pip install anthropic --quiet

# Importujte modul hints z balíčku utils
import os
import sys
module_path = ".."
sys.path.append(os.path.abspath(module_path))
from utils import hints

# Importujte vestavěnou knihovnu regulárních výrazů v Pythonu
import re
from anthropic import AnthropicBedrock

%store -r MODEL_NAME
%store -r AWS_REGION

client = AnthropicBedrock(aws_region=AWS_REGION)

def get_completion(prompt, system=''):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": prompt}
        ],
        system=system
    )
    return message.content[0].text

---

## Lekce

Pokračujeme v tématu, že Claude nemá žádný kontext kromě toho, co řeknete, je někdy důležité **přimět Clauda, aby zaujal specifickou roli (včetně veškerého potřebného kontextu)**. Toto je také známé jako role prompting. Čím více detailů k roli, tím lépe.

**Připravení Clauda s rolí může zlepšit jeho výkon** v různých oblastech, od psaní po programování až po shrnování. Je to jako když lidem někdy pomůže, když jim řeknete, aby "mysleli jako ______". Role prompting může také změnit styl, tón a způsob, jakým Claude reaguje.

**Poznámka:** Role prompting může probíhat buď v system prompt, nebo jako součást uživatelské zprávy.

### Příklady

V níže uvedeném příkladu vidíme, že bez role promptingu poskytuje Claude **přímou a nestylizovanou odpověď**, když je požádán o jednovětnou perspektivu na skateboarding.

Když však Claudea připravíme na roli kočky, Claudeova perspektiva se změní, a tím se **Claudeův tón odpovědi, styl a obsah přizpůsobí nové roli**.

**Poznámka:** Bonusová technika, kterou můžete použít, je **poskytnout Claudovi kontext o jeho zamýšleném publiku**. Níže jsme mohli upravit prompt tak, aby Claudovi také řekl, s kým by měl mluvit. "Jsi kočka" vyvolá zcela jinou odpověď než "jsi kočka mluvící k davu skateboardistů".

Zde je prompt bez role promptingu v systémovém promptu:

In [ ]:
```python
# Prompt
PROMPT = "In one sentence, what do you think about skateboarding?"

# Vytiskni Claudeovu odpověď
print(get_completion(PROMPT))
```

Zde je stejná otázka uživatele, ale s použitím role promptu.

In [ ]:
```python
# Systémový prompt
SYSTEM_PROMPT = "You are a cat."

# Prompt
PROMPT = "In one sentence, what do you think about skateboarding?"

# Vytiskni odpověď od Claude
print(get_completion(PROMPT, SYSTEM_PROMPT))
```

Můžete použít role prompting jako způsob, jak přimět Claude, aby napodoboval určité styly psaní, mluvil určitým hlasem nebo řídil složitost svých odpovědí. **Role prompting může také zlepšit schopnost Claude vykonávat matematické nebo logické úkoly.**

Například v níže uvedeném příkladu existuje jednoznačně správná odpověď, kterou je ano. Nicméně, Claude se mýlí a myslí si, že mu chybí informace, což není pravda:

In [ ]:
```python
# Prompt
PROMPT = "Jack is looking at Anne. Anne is looking at George. Jack is married, George is not, and we don’t know if Anne is married. Is a married person looking at an unmarried person?"

# Vytisknout Claudeovu odpověď
print(get_completion(PROMPT))
```

Nyní, co kdybychom **nastavili Claudea, aby fungoval jako logický bot**? Jak to změní Claudeovu odpověď?

Ukazuje se, že s tímto novým přiřazením role to Claude zvládne správně. (I když ne nutně ze všech správných důvodů)

In [ ]:
```python
# Systémový prompt
SYSTEM_PROMPT = "You are a logic bot designed to answer complex logic problems."

# Prompt
PROMPT = "Jack se dívá na Anne. Anne se dívá na George. Jack je ženatý, George není, a nevíme, zda je Anne vdaná. Dívá se ženatá osoba na neženatou osobu?"

# Vytisknout Claudeovu odpověď
print(get_completion(PROMPT, SYSTEM_PROMPT))
```

**Poznámka:** Co se během tohoto kurzu naučíte, je, že existuje **mnoho technik prompt engineeringu, které můžete použít k dosažení podobných výsledků**. Které techniky použijete, záleží na vás a vašich preferencích! Doporučujeme vám **experimentovat a najít si svůj vlastní styl prompt engineeringu**.

Pokud byste chtěli experimentovat s prompty z lekce, aniž byste měnili jakýkoli obsah výše, přejděte až na konec notebooku lekce a navštivte [**Example Playground**](#example-playground).

## Cvičení
- [Cvičení 3.1 - Oprava matematiky](#exercise-31---math-correction)

### Cvičení 3.1 - Oprava matematiky
V některých případech **může mít Claude problémy s matematikou**, dokonce i s jednoduchou matematikou. Níže Claude nesprávně posoudí matematický problém jako správně vyřešený, i když je ve druhém kroku zjevná aritmetická chyba. Všimněte si, že Claude chybu skutečně zachytí při postupném procházení, ale nedospěje k závěru, že celkové řešení je špatné.

Upravte `PROMPT` a / nebo `SYSTEM_PROMPT`, aby Claude ohodnotil řešení jako `nesprávně` vyřešené, místo správně vyřešeného.

In [ ]:
```python
# Systemový prompt - pokud nechcete použít systémový prompt, můžete tuto proměnnou nechat nastavenou na prázdný řetězec
SYSTEM_PROMPT = ""

# Prompt
PROMPT = """Is this equation solved correctly below?

2x - 3 = 9
2x = 6
x = 3"""

# Získat odpověď od Claude
response = get_completion(PROMPT, SYSTEM_PROMPT)

# Funkce pro hodnocení správnosti cvičení
def grade_exercise(text):
    if "incorrect" in text or "not correct" in text.lower():
        return True
    else:
        return False

# Vytisknout odpověď od Claude a odpovídající hodnocení
print(response)
print("\n--------------------------- HODNOCENÍ ---------------------------")
print("Toto cvičení bylo správně vyřešeno:", grade_exercise(response))
```

❓ Pokud chcete nápovědu, spusťte buňku níže!

In [ ]:
print(hints.exercise_3_1_hint)

### Gratulujeme!

Pokud jste vyřešili všechny úkoly až do tohoto bodu, jste připraveni přejít k další kapitole. Šťastné promptování!

---

## Příklad hřiště

Toto je prostor, kde můžete volně experimentovat s příklady promptů uvedenými v této lekci a upravovat prompty, abyste viděli, jak to může ovlivnit odpovědi Claude.

In [ ]:
```python
# Prompt
PROMPT = "In one sentence, what do you think about skateboarding?"

# Vytisknout odpověď od Claude
print(get_completion(PROMPT))
```

In [ ]:
```python
# Systémový prompt
SYSTEM_PROMPT = "You are a cat."

# Prompt
PROMPT = "In one sentence, what do you think about skateboarding?"

# Vytisknout odpověď od Claude
print(get_completion(PROMPT, SYSTEM_PROMPT))
```

In [ ]:
```python
# Prompt
PROMPT = "Jack is looking at Anne. Anne is looking at George. Jack is married, George is not, and we don’t know if Anne is married. Is a married person looking at an unmarried person?"

# Vytisknout Claudeovu odpověď
print(get_completion(PROMPT))
```

In [ ]:
```python
# Systémový prompt
SYSTEM_PROMPT = "You are a logic bot designed to answer complex logic problems."

# Prompt
PROMPT = "Jack se dívá na Anne. Anne se dívá na George. Jack je ženatý, George není, a nevíme, zda je Anne vdaná. Dívá se ženatý člověk na neženatého člověka?"

# Vytisknout odpověď od Claude
print(get_completion(PROMPT, SYSTEM_PROMPT))
```